# RQ2 — Regionaler Vergleich nach Stadtgröße

Vergleicht Hitzewellen (Häufigkeit, Intensität, Dauer) zwischen Großstadt,
Mittelstadt, Kleinstadt und Landgemeinde — jeweils im selben Bundesland,
für einen regionalen Stadt-Land-Vergleich.

Baut direkt auf dem RQ1-Notebook auf: Open-Meteo-Client sowie Fetch- und
Hitzewellen-Erkennungsfunktionen sind **identisch übernommen** (gleiche
Methodik = vergleichbare Ergebnisse zwischen den Größenklassen). Die
Großstadt-Kategorie wird direkt aus `data/processed/heatwaves.csv`
(RQ1-Ergebnis) übernommen statt erneut berechnet.

Kategorien (nach Projektdefinition):

| Kategorie      | Einwohner         |
|----------------|-------------------|
| Großstadt      | ≥ 100.000         |
| Mittelstadt    | 20.000 – 100.000  |
| Kleinstadt     |  5.000 –  20.000  |
| Landgemeinde   | < 5.000           |

Die Großstadt-**Ankerliste** bleibt bei ≥150.000 EW (wie in RQ1, aus
Gründen der API-Last und um die bestehenden Ergebnisse wiederzuverwenden).

**Zelle 1 — Alle deutschen Gemeinden laden**

Holt (einmalig, dann lokal gecacht) alle ~11.000 deutschen Gemeinden von
Wikidata (Klasse *Gemeinde in Deutschland*, `Q262166`) inkl. Einwohnerzahl,
Koordinaten und Bundesland. Das ist der komplette Kandidatenpool für die
Mittelstadt/Kleinstadt/Landgemeinde-Zuordnung — die Großstadtliste
(≥150.000) ist einfach ein Filter auf denselben Datensatz.

In [11]:
import os
if os.path.exists("data/raw/municipalities_de.csv"):
    os.remove("data/raw/municipalities_de.csv")

In [12]:
# Alle deutschen Gemeinden inkl. Einwohner, Koordinaten, Bundesland (Wikidata-SPARQL-API)
# -- pro Bundesland einzeln abgefragt, um den 60s-Timeout des WDQS-Endpoints zu vermeiden
import os
import time
import requests
import pandas as pd
import numpy as np

MUNI_CACHE = "data/raw/municipalities_de.csv"
SPARQL_URL = "https://query.wikidata.org/sparql"
HEADERS = {"Accept": "application/sparql-results+json", "User-Agent": "student-project/1.0"}


def sparql_query(query, retries=3, pause=5):
    """Fuehrt eine SPARQL-Anfrage aus, mit Wiederholungsversuchen bei 5xx-Fehlern
    (der WDQS-Endpoint antwortet bei zu grossen/langsamen Queries mit 504 --
    kleinere, gezielte Abfragen sind der eigentliche Fix, ein Retry faengt aber
    zusaetzlich noch vereinzelte, transiente Fehler ab)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(SPARQL_URL, params={"query": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            return r.json()["results"]["bindings"]
        except (requests.exceptions.HTTPError, requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            if attempt == retries:
                raise
            print(f"    Fehler ({e}), Versuch {attempt}/{retries}, warte {pause}s...")
            time.sleep(pause)


def get_bundeslaender():
    """Kleine, schnelle Abfrage: alle 16 deutschen Bundeslaender mit QID."""
    query = """
    SELECT DISTINCT ?bundesland ?bundeslandLabel WHERE {
      ?bundesland wdt:P31 wd:Q1221156 .
      SERVICE wikibase:label { bd:serviceParam wikibase:language "de,en". }
    }
    """
    rows = sparql_query(query)
    return [(row["bundesland"]["value"].split("/")[-1], row["bundeslandLabel"]["value"]) for row in rows]


def get_municipalities_for_bundesland(bundesland_qid):
    """Alle Gemeinden EINES Bundeslands.

    ZWEI separate Fixes gegenueber der Vorversion (die nur 48 statt ~11.000
    Gemeinden lieferte):

    1) Subklassen-Abschluss auf der TYP-Seite: `wdt:P31/wdt:P279* wd:Q262166`
       statt `wdt:P31 wd:Q262166` (exakter Match). Die meisten deutschen
       Gemeinden sind vermutlich nicht direkt als "Gemeinde in Deutschland"
       (Q262166) getaggt, sondern als spezifischere Unterklasse (z.B.
       "Gemeinde in Bayern"), die per P279 eine Subklasse von Q262166 ist --
       exakt das gleiche Muster wie in RQ1 Zelle 1 (`wdt:P31/wdt:P279* wd:Q515`
       fuer Staedte), dort aus demselben Grund verwendet. Das ist der
       vermutliche Hauptgrund fuer die 48 statt ~11.000 Treffer.

    2) Hop-Tiefe der Verwaltungskette auf 3 statt 2 erweitert (statt eines
       offenen `+`-Pfads, der wieder den 502/504-Timeout ausloesen wuerde):
       manche Bundeslaender haben zusaetzliche Zwischenebenen (Verbandsgemeinde/
       Amt/Samtgemeinde zwischen Gemeinde und Landkreis, z.B. in
       Rheinland-Pfalz), die eine reine 2-Hop-Kette (Gemeinde -> Landkreis ->
       Bundesland) verfehlen wuerde. Weiterhin klein und bounded, also
       weiterhin kein Performance-Risiko.
    """
    query = f"""
    SELECT DISTINCT ?gemeindeLabel ?population ?coord WHERE {{
      ?gemeinde wdt:P31/wdt:P279* wd:Q262166 ;
                wdt:P1082 ?population ;
                wdt:P625 ?coord .
      {{ ?gemeinde wdt:P131 wd:{bundesland_qid} . }}
      UNION
      {{ ?gemeinde wdt:P131 ?l1 . ?l1 wdt:P131 wd:{bundesland_qid} . }}
      UNION
      {{ ?gemeinde wdt:P131 ?l1 . ?l1 wdt:P131 ?l2 . ?l2 wdt:P131 wd:{bundesland_qid} . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "de,en". }}
    }}
    """
    return sparql_query(query)


def get_all_municipalities(use_cache=True):
    if use_cache and os.path.exists(MUNI_CACHE):
        return pd.read_csv(MUNI_CACHE)

    bundeslaender = get_bundeslaender()
    print(f"{len(bundeslaender)} Bundeslaender gefunden, frage jetzt einzeln ab...")

    all_rows = []
    failed_bundeslaender = []
    for qid, name in bundeslaender:
        print(f"  {name} ({qid})...")
        try:
            rows = get_municipalities_for_bundesland(qid)
        except Exception as e:
            # Ein dauerhaft scheiterndes Bundesland soll den gesamten Lauf nicht
            # abbrechen -- das Matching in Zelle 2 hat fuer genau diesen Fall
            # bereits einen bundesweiten Fallback eingebaut.
            print(f"    UEBERSPRUNGEN wegen Fehler: {e}")
            failed_bundeslaender.append(name)
            continue
        for row in rows:
            try:
                # Koordinaten kommen als Text "Point(lon lat)" -> aufsplitten
                lon, lat = row["coord"]["value"].replace("Point(", "").replace(")", "").split(" ")
                all_rows.append({
                    "city": row["gemeindeLabel"]["value"],
                    "population": int(float(row["population"]["value"])),
                    "lat": float(lat), "lon": float(lon),
                    "bundesland": name,
                })
            except (KeyError, ValueError):
                continue  # unvollstaendige Zeile ueberspringen
        print(f"    -> {len(rows)} Gemeinden")
        time.sleep(1)  # Endpoint nicht zuballern

    if failed_bundeslaender:
        print(f"\nWARNUNG: nicht ladbar: {failed_bundeslaender}")
        print("(betroffene Grossstaedte nutzen beim Matching automatisch den bundesweiten Fallback)")

    df = pd.DataFrame(all_rows).drop_duplicates(subset=["city", "bundesland"])
    df = df[df["population"] > 0].reset_index(drop=True)

    os.makedirs("data/raw", exist_ok=True)
    df.to_csv(MUNI_CACHE, index=False)
    return df


pool_df = get_all_municipalities()
print(f"\nAnzahl Gemeinden gesamt: {len(pool_df)}")

16 Bundeslaender gefunden, frage jetzt einzeln ab...
  Berlin (Q64)...
    -> 0 Gemeinden
  Hamburg (Q1055)...
    -> 0 Gemeinden
  Saarland (Q1201)...
    -> 52 Gemeinden
  Bayern (Q980)...
    -> 2266 Gemeinden
  Baden-Württemberg (Q985)...
    -> 1209 Gemeinden
  Brandenburg (Q1208)...
    -> 427 Gemeinden
  Hessen (Q1199)...
    -> 425 Gemeinden
  Mecklenburg-Vorpommern (Q1196)...
    -> 731 Gemeinden
  Nordrhein-Westfalen (Q1198)...
    -> 397 Gemeinden
  Niedersachsen (Q1197)...
    -> 1003 Gemeinden
  Schleswig-Holstein (Q1194)...
    -> 1116 Gemeinden
  Sachsen (Q1202)...
    -> 477 Gemeinden
  Rheinland-Pfalz (Q1200)...
    -> 2324 Gemeinden
  Sachsen-Anhalt (Q1206)...
    -> 278 Gemeinden
  Thüringen (Q1205)...
    -> 858 Gemeinden
  Freie Hansestadt Bremen (Q1209)...
    -> 2 Gemeinden

Anzahl Gemeinden gesamt: 11010


**Zelle 2 — Kategorien zuordnen und regionale Matches finden**

Klassifiziert jede Gemeinde nach Einwohnerzahl und sucht für jede
Großstadt (≥150.000 EW) den nächstgelegenen Mittelstadt-, Kleinstadt-
und Landgemeinde-Kandidaten im selben Bundesland (Luftlinie). Fallback:
bundesweit (max. 200 km), falls im Bundesland nichts (mehr) übrig ist —
relevant v.a. für Berlin/Hamburg/Bremen. `avoid_reuse=True` verhindert,
dass derselbe kleine Ort zwei Großstädten zugeordnet wird.

In [13]:
def classify(pop):
    if pop >= 100_000:
        return "Grossstadt"
    elif pop >= 20_000:
        return "Mittelstadt"
    elif pop >= 5_000:
        return "Kleinstadt"
    else:
        return "Landgemeinde"

pool_df["category"] = pool_df["population"].apply(classify)
print(pool_df["category"].value_counts())


def haversine_km(lat1, lon1, lat2, lon2):
    """Luftlinien-Distanz in km zwischen einem Punkt und (Vektoren von) Punkten."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def find_regional_matches(grossstaedte, pool, categories=("Mittelstadt", "Kleinstadt", "Landgemeinde"),
                           avoid_reuse=True, max_fallback_km=200.0):
    used = set()
    rows = []
    for _, gs in grossstaedte.iterrows():
        row = {"grossstadt": gs["city"], "bundesland": gs["bundesland"], "gs_population": gs["population"]}
        same_land = pool[pool["bundesland"] == gs["bundesland"]].copy()
        same_land["distance_km"] = haversine_km(gs["lat"], gs["lon"], same_land["lat"].values, same_land["lon"].values)

        for cat in categories:
            cands = same_land[same_land["category"] == cat].sort_values("distance_km")
            if avoid_reuse:
                cands = cands[~cands["city"].isin(used)]

            fallback = False
            if cands.empty:
                # z.B. Berlin/Hamburg/Bremen haben kaum kleinere Gemeinden im eigenen Bundesland
                nation = pool[pool["category"] == cat].copy()
                nation["distance_km"] = haversine_km(gs["lat"], gs["lon"], nation["lat"].values, nation["lon"].values)
                nation = nation[nation["distance_km"] <= max_fallback_km]
                if avoid_reuse:
                    nation = nation[~nation["city"].isin(used)]
                cands = nation.sort_values("distance_km")
                fallback = True

            if not cands.empty:
                m = cands.iloc[0]
                row[f"{cat}_name"] = m["city"]
                row[f"{cat}_population"] = m["population"]
                row[f"{cat}_lat"] = m["lat"]
                row[f"{cat}_lon"] = m["lon"]
                row[f"{cat}_distance_km"] = round(float(m["distance_km"]), 1)
                row[f"{cat}_fallback"] = fallback
                if avoid_reuse:
                    used.add(m["city"])
            else:
                row[f"{cat}_name"] = None
                print(f"  WARNUNG: kein Match fuer {gs['city']} / {cat}")
        rows.append(row)
    return pd.DataFrame(rows)


grossstaedte_df = pool_df[pool_df["population"] >= 150_000].reset_index(drop=True)
print(f"Grossstaedte (>=150.000 EW): {len(grossstaedte_df)}")
assert len(grossstaedte_df) > 0, (
    "Keine Grossstaedte gefunden -- pool_df hat vermutlich zu wenige Zeilen "
    "(pruefe 'Anzahl Gemeinden gesamt' aus Zelle 1; sollte ~11.000 sein, nicht "
    "48 -- ggf. data/raw/municipalities_de.csv loeschen und Zelle 1 neu laufen lassen)."
)

matches_df = find_regional_matches(grossstaedte_df, pool_df)
os.makedirs("data/processed", exist_ok=True)
matches_df.to_csv("data/processed/regional_matches.csv", index=False)
matches_df[["grossstadt", "Mittelstadt_name", "Kleinstadt_name", "Landgemeinde_name"]]

category
Landgemeinde    8019
Kleinstadt      2283
Mittelstadt      628
Grossstadt        80
Name: count, dtype: int64
Grossstaedte (>=150.000 EW): 54


,grossstadt,Mittelstadt_name,Kleinstadt_name,Landgemeinde_name
0,Saarbrücken,St. Ingbert,Sulzbach/Saar,Hornbach
1,München,Unterhaching,Unterföhring,Baierbrunn
2,Augsburg,Gersthofen,Stadtbergen,Aystetten
3,Regensburg,Schwandorf,Lappersdorf,Pettendorf
4,Nürnberg,Zirndorf,Stein,Kalchreuth
5,Heidelberg,Leimen,Dossenheim,Gaiberg
6,Freiburg im Breisgau,Waldkirch,Merzhausen,Au
7,Mannheim,Schwetzingen,Ilvesheim,Wilhelmsfeld
8,Stuttgart,Korntal-Münchingen,Gerlingen,Affalterbach
9,Karlsruhe,Stutensee,Eggenstein-Leopoldshafen,Au am Rhein


**Zelle 3 — Standortliste für neue Wetterabfragen**

Nur die neu gefundenen Mittelstädte, Kleinstädte und Landgemeinden
brauchen frische Wetterdaten — die Großstadt-Werte liegen bereits aus
RQ1 vor (`data/processed/heatwaves.csv`) und werden weiter unten
wiederverwendet statt erneut abgefragt.

In [ ]:
# matches_df (breit) -> Langformat: eine Zeile pro neuem Standort
new_rows = []
for _, r in matches_df.iterrows():
    for cat in ("Mittelstadt", "Kleinstadt", "Landgemeinde"):
        name = r.get(f"{cat}_name")
        if pd.isna(name):
            continue
        new_rows.append({
            "city": name,
            "category": cat,
            "match_group": r["grossstadt"],   # verweist auf die zugehoerige Grossstadt
            "population": r[f"{cat}_population"],
            "lat": r[f"{cat}_lat"],
            "lon": r[f"{cat}_lon"],
        })

new_locations_df = pd.DataFrame(new_rows).drop_duplicates(subset="city").reset_index(drop=True)
print(f"Neue Standorte (Wetterdaten fehlen noch): {len(new_locations_df)}")
new_locations_df["category"].value_counts()

**Zelle 4 — Open-Meteo-Client**

Identisch zu RQ1: FlatBuffers-Client + persistenter SQLite-Cache.
Wichtig: gleiche Cache-Datei (`openmeteo_cache.sqlite`), also im
gleichen Arbeitsverzeichnis wie das RQ1-Notebook laufen lassen.

In [ ]:
# Open-Meteo-Client: FlatBuffers-Client + persistenter Cache + eigene Retry-Logik (inkl. 429)
import openmeteo_requests
import requests_cache
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_client():
    cache_session = requests_cache.CachedSession(
        "openmeteo_cache",      # Datei: openmeteo_cache.sqlite
        expire_after=-1,         # nie ablaufen -- Klimadaten aendern sich nicht rueckwirkend
        allowable_codes=[200],   # 429/Fehler werden NIE gecacht
    )
    retry = Retry(
        total=6, backoff_factor=3,                       # 3s, 6s, 12s, 24s, 48s, 96s
        status_forcelist=[429, 500, 502, 503, 504],       # 429 explizit dabei
        allowed_methods=["GET"],
    )
    cache_session.mount("https://", HTTPAdapter(max_retries=retry))
    cache_session.headers.update({"User-Agent": "student-project/1.0"})
    return openmeteo_requests.Client(session=cache_session)

CLIENT = make_client()

**Zelle 5 — Wetterdaten- und Hitzewellen-Funktionen**

1:1 aus RQ1 übernommen (gleiche Methodik = vergleichbare Ergebnisse
zwischen den Größenklassen): `fetch_tmax_batch`, `fetch_tmax_range`,
`compute_doy_thresholds`, `find_dwd_heatwaves`,
`analyze_city_heatwaves_from_data`.

In [ ]:
import time

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

def fetch_tmax_batch(client, coords, start_date, end_date, batch_size=15, pause=5):
    """Holt Tages-Maximaltemperaturen fuer mehrere Staedte gleichzeitig."""
    dfs = []
    n_batches = -(-len(coords) // batch_size)  # aufrunden

    for b in range(n_batches):
        i = b * batch_size
        chunk = coords[i:i + batch_size]  # ein Batch von Staedten

        params = {
            "latitude": [lat for lat, lon in chunk],
            "longitude": [lon for lat, lon in chunk],
            "start_date": start_date, "end_date": end_date,
            "daily": ["temperature_2m_max"],
            "timezone": "Europe/Berlin",
        }
        try:
            responses = client.weather_api(ARCHIVE_URL, params=params)  # immer eine Liste
        except Exception as e:
            raise RuntimeError(f"API-Fehler bei Batch {b+1}/{n_batches}: {e}") from e

        for response in responses:
            daily = response.Daily()
            tmax = daily.Variables(0).ValuesAsNumpy()  # 0 = einzige angeforderte Variable
            dates = pd.date_range(
                start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=daily.Interval()),
                inclusive="left",
            ).tz_localize(None)  # Zeitzone abstreifen, reine Kalendertage reichen
            dfs.append(pd.DataFrame({"date": dates, "tmax": tmax}))

        print(f"  Batch {b+1}/{n_batches} ok ({len(chunk)} Staedte)")
        time.sleep(pause)

    return dfs


def fetch_tmax_range(client, coords, start_date, end_date, batch_size=15, year_chunk=5, pause=5):
    """Holt Tmax ueber einen langen Zeitraum, aufgeteilt in Staedte-Batches
    UND Jahres-Bloecke -- jede einzelne Anfrage bleibt klein, Fortschritt
    ist granular im HTTP-Cache gesichert."""
    start_year, end_year = int(start_date[:4]), int(end_date[:4])

    year_ranges = []
    y = start_year
    while y <= end_year:
        y2 = min(y + year_chunk - 1, end_year)
        year_ranges.append((f"{y}-01-01", f"{y2}-12-31"))
        y = y2 + 1

    all_parts = {i: [] for i in range(len(coords))}
    for chunk_start, chunk_end in year_ranges:
        print(f"Zeitraum {chunk_start[:4]}-{chunk_end[:4]}:")
        dfs = fetch_tmax_batch(client, coords, chunk_start, chunk_end, batch_size=batch_size, pause=pause)
        for i, df in enumerate(dfs):
            all_parts[i].append(df)
        time.sleep(pause)  # zusaetzlicher Puffer zwischen Zeitbloecken

    return [pd.concat(all_parts[i], ignore_index=True).sort_values("date").reset_index(drop=True)
            for i in range(len(coords))]


def compute_doy_thresholds(ref_df, window=15, percentile=98):
    """Schwellenwert pro Kalendertag: 98. Perzentil aus +-15 Tagen der Referenzperiode."""
    doy = ref_df["doy"].to_numpy()
    tmax = ref_df["tmax"].to_numpy()
    thresholds = {}

    for d in range(1, 366):
        dist = np.abs(doy - d)
        dist = np.minimum(dist, 365 - dist)  # Jahreswechsel beruecksichtigen
        thresholds[d] = np.percentile(tmax[dist <= window], percentile)

    return thresholds


def find_dwd_heatwaves(df, thresholds, min_days=3, fixed_threshold=28.0):
    """Hitzewelle = mind. 3 Tage in Folge ueber Klima-Schwelle UND ueber 28C."""
    df = df.copy()
    df["doy"] = df["date"].dt.dayofyear.clip(upper=365)
    df["threshold"] = df["doy"].map(thresholds)
    df["is_hot"] = (df["tmax"] > df["threshold"]) & (df["tmax"] > fixed_threshold)

    heatwaves, start = [], None
    for i, row in df.iterrows():
        if row["is_hot"] and start is None:
            start = i
        elif not row["is_hot"] and start is not None:
            segment = df.loc[start:i - 1]
            if len(segment) >= min_days:
                heatwaves.append({
                    "start": segment["date"].iloc[0].date(),
                    "end": segment["date"].iloc[-1].date(),
                    "duration_days": len(segment),
                    "max_temp": round(segment["tmax"].max(), 1),
                    "avg_temp": round(segment["tmax"].mean(), 1)
                })
            start = None

    if start is not None:
        segment = df.loc[start:]
        if len(segment) >= min_days:
            heatwaves.append({
                "start": segment["date"].iloc[0].date(),
                "end": segment["date"].iloc[-1].date(),
                "duration_days": len(segment),
                "max_temp": round(segment["tmax"].max(), 1),
                "avg_temp": round(segment["tmax"].mean(), 1)
            })

    return heatwaves


def analyze_city_heatwaves_from_data(ref_df, analysis_df):
    """Verbindet Schwellenwert-Berechnung und Hitzewellen-Suche fuer eine Stadt."""
    ref_df = ref_df.copy()
    ref_df["doy"] = ref_df["date"].dt.dayofyear.clip(upper=365)
    thresholds = compute_doy_thresholds(ref_df)
    return find_dwd_heatwaves(analysis_df, thresholds)

**Zelle 6 — Kurztest** (prüft, dass die API auch für kleine Orte sauber antwortet)

In [ ]:
coords_new = list(zip(new_locations_df["lat"], new_locations_df["lon"]))
test = fetch_tmax_batch(CLIENT, coords_new[:3], "2025-01-01", "2025-01-31")
[len(df) for df in test]  # sollte 3x ~31 ergeben, auch fuer kleine Orte

**Zelle 7 — Referenzperiode (1961–1990) und Analysezeitraum (1980–2025) laden**

Nur für die neuen Standorte — gleiche Zeiträume wie in RQ1, damit die
Klimaschwellenwerte methodisch identisch berechnet werden.

In [ ]:
#Fuer cache aufrufe DAS HIER NEHMEN!!
ref_dfs_new = fetch_tmax_range(CLIENT, coords_new, "1961-01-01", "1990-12-31", batch_size=15, year_chunk=10, pause=1)

In [ ]:
analysis_dfs_new = fetch_tmax_range(CLIENT, coords_new, "1980-01-01", "2025-12-31", batch_size=15, year_chunk=5, pause=1)

**Zelle 8 — Hitzewellen für die neuen Standorte berechnen**

In [ ]:
results_new = []
failed_new = []
for idx, row in new_locations_df.iterrows():
    try:
        heatwaves = analyze_city_heatwaves_from_data(ref_dfs_new[idx], analysis_dfs_new[idx])
        for hw in heatwaves:
            results_new.append({
                "city": row["city"], "category": row["category"],
                "match_group": row["match_group"], "population": row["population"],
                **hw
            })
    except Exception as e:
        print(f"  Uebersprungen wegen Fehler ({row['city']}): {e}")
        failed_new.append(row["city"])

results_new_df = pd.DataFrame(results_new)
if not results_new_df.empty:
    results_new_df["year"] = pd.to_datetime(results_new_df["start"]).dt.year
else:
    # Falls an keinem neuen Standort eine Hitzewelle gefunden wurde (z.B. bei
    # lauter fehlgeschlagenen Fetches) -- Spalte trotzdem anlegen, sonst crasht
    # der spaetere Merge/Concat.
    results_new_df["year"] = pd.Series(dtype="int64")
print(f"Hitzewellen an neuen Standorten: {len(results_new_df)}")
print(f"Fehlgeschlagen: {failed_new}")

**Zelle 9 — Mit den Großstadt-Ergebnissen aus RQ1 zusammenführen**

`data/processed/heatwaves.csv` enthält bereits alle Hitzewellen der 57
Großstädte aus RQ1 — wird hier direkt wiederverwendet statt neu berechnet.

In [ ]:
results_gs_df = pd.read_csv("data/processed/heatwaves.csv")
results_gs_df["category"] = "Grossstadt"
results_gs_df["match_group"] = results_gs_df["city"]  # Grossstadt ist ihr eigener Anker

# Sanity-Check: passen die Grossstadt-Namen aus dem Matching zu denen aus RQ1?
matched_gs_names = set(matches_df["grossstadt"])
missing = matched_gs_names - set(results_gs_df["city"])
if missing:
    print(f"WARNUNG: keine RQ1-Daten fuer: {missing}")

results_all_df = pd.concat([results_gs_df, results_new_df], ignore_index=True)
results_all_df.to_csv("data/processed/heatwaves_rq2_all.csv", index=False)
print(f"Gesamt: {len(results_all_df)} Hitzewellen ueber {results_all_df['category'].nunique()} Kategorien")
results_all_df.groupby("category").size()

**Zelle 10 — Aggregation pro Standort (gesamter Zeitraum 1980–2025)**

In [ ]:
city_summary = results_all_df.groupby(["city", "category", "match_group"]).agg(
    n_heatwaves=("start", "count"),
    avg_max_temp=("max_temp", "mean"),
    avg_avg_temp=("avg_temp", "mean"),
    avg_duration=("duration_days", "mean"),
).reset_index()

# Orte ganz ohne Hitzewelle fehlen sonst komplett -> mit 0 auffuellen
all_locations = pd.concat([
    grossstaedte_df[["city"]].assign(category="Grossstadt", match_group=grossstaedte_df["city"]),
    new_locations_df[["city", "category", "match_group"]],
])
city_summary = all_locations.merge(city_summary, on=["city", "category", "match_group"], how="left")
city_summary["n_heatwaves"] = city_summary["n_heatwaves"].fillna(0).astype(int)

city_summary.to_csv("data/processed/city_summary_rq2.csv", index=False)
city_summary.groupby("category")[["n_heatwaves", "avg_max_temp", "avg_avg_temp", "avg_duration"]].mean()

**Zelle 11 — Boxplots nach Städtegröße**

In [ ]:
import matplotlib.pyplot as plt

order = ["Grossstadt", "Mittelstadt", "Kleinstadt", "Landgemeinde"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def boxplot_by_category(ax, col, title, ylabel):
    data = [city_summary[city_summary["category"] == c][col].dropna() for c in order]
    ax.boxplot(data, labels=order)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)

boxplot_by_category(axes[0,0], "n_heatwaves", "Anzahl Hitzewellen (1980-2025)", "Anzahl")
boxplot_by_category(axes[0,1], "avg_max_temp", "Oe Spitzentemperatur", "Temperatur (C)")
boxplot_by_category(axes[1,0], "avg_avg_temp", "Oe Durchschnittstemperatur", "Temperatur (C)")
boxplot_by_category(axes[1,1], "avg_duration", "Oe Dauer pro Hitzewelle", "Tage")

plt.tight_layout()
plt.savefig("data/processed/rq2_comparison_boxplots.png", dpi=150)
plt.show()

**Zelle 12 — Statistischer Vergleich: Großstadt vs. kleinere Kategorien (gepaart pro Region)**

Da jede Mittelstadt/Kleinstadt/Landgemeinde einer bestimmten Großstadt
zugeordnet ist (`match_group`), sind die Beobachtungen nicht unabhängig
— ein gepaarter Test (Wilcoxon signed-rank) ist hier angemessener als
z.B. eine unabhängige ANOVA über alle Orte.

In [ ]:
from scipy import stats

pivot = city_summary.pivot_table(index="match_group", columns="category",
                                   values=["n_heatwaves", "avg_max_temp", "avg_avg_temp", "avg_duration"])

for metric, label in [("n_heatwaves", "Haeufigkeit"), ("avg_max_temp", "Spitzentemp."),
                       ("avg_avg_temp", "Durchschnittstemp."), ("avg_duration", "Dauer")]:
    print(f"\n--- {label} ---")
    for cat in ["Mittelstadt", "Kleinstadt", "Landgemeinde"]:
        pair = pivot[metric][["Grossstadt", cat]].dropna()
        if len(pair) < 5:
            print(f"  Grossstadt vs {cat}: zu wenige gepaarte Beobachtungen (n={len(pair)})")
            continue
        stat, p = stats.wilcoxon(pair["Grossstadt"], pair[cat])
        signif = "signifikant" if p < 0.05 else "nicht signifikant"
        diff = (pair["Grossstadt"] - pair[cat]).mean()
        print(f"  Grossstadt vs {cat}: Oe-Differenz={diff:+.2f}, p={p:.4f} ({signif}), n={len(pair)}")